In [ ]:
#|default_exp sgd

# Load deps

In [ ]:
# !pip install -q torcheval
# # or
# !uv add torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# from pathlib import Path
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
#|export
import torch
from src.learner import Callback

In [ ]:
import pickle,gzip,math,os,time,shutil,torch,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter,itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager

import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder

from src.utils import set_seed
from src.datasets import inplace, DataLoaders
from src.learner import (
    DeviceCB, MetricsCB, ProgressCB,
    LRFinderCB, TrainLearner, MomentumLearner,
    SingleBatchCB
)
from src.activations import ActivationStats
from src.init import GeneralRelu, init_weights, get_model
from src.utils import store_attr

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
set_seed(42)
ds_name = "fashion_mnist"
ds_xl,ds_yl = 'image','label'
bs = 1024
num_dl_workers = 2

# Load data

In [ ]:
dsd = load_dataset(ds_name)

train_input_avg = dsd["train"]\
  .map(
      lambda x: {
          "input_mean": [
              TF.to_tensor(image).mean() for image in x[ds_xl]
          ],
          "input_std": [
              TF.to_tensor(image).std() for image in x[ds_xl]
          ]
      },
      batched=True,
      batch_size=10000,
      remove_columns=dsd["train"].column_names,
  )

input_mean_df = train_input_avg.to_pandas().mean()
xmean = input_mean_df["input_mean"]
xstd = input_mean_df["input_std"]

In [ ]:
@inplace
def transformi(b): b[ds_xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[ds_xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)

# Set up callbacks

In [ ]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(lambda mod: isinstance(mod, GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
lrf_cbs = [DeviceCB(), LRFinderCB()]

# Optimizers

## SGD

In [ ]:
class SGD:
    def __init__(self, params, lr, wd=0.):
        params = list(params)
        store_attr()
        self.i = 0

    def step(self):
        with torch.no_grad():
            for p in self.params:
                self.reg_step(p)
                self.opt_step(p)
        self.i +=1

    def opt_step(self, p): p -= p.grad * self.lr
    def reg_step(self, p):
        if self.wd != 0: p *= 1 - self.lr*self.wd

    def zero_grad(self):
        for p in self.params: p.grad.data.zero_()

In [ ]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=0.4, cbs=cbs, opt_func=SGD)
learn.fit(3)

## Momentum

In [ ]:
xs = torch.linspace(-4, 4, 100)
ys = 1 - (xs/3) ** 2 + torch.randn(100) * 0.1
_,axs = plt.subplots(2,2, figsize=(8,6))
betas = [0.5,0.7,0.9,0.99]
for beta,ax in zip(betas, axs.flatten()):
    ax.scatter(xs,ys)
    avg,res = 0,[]
    for yi in ys:
        avg = beta*avg + (1-beta)*yi
        res.append(avg)
    ax.plot(xs,np.array(res), color='red');
    ax.set_title(f'beta={beta}')

In [ ]:
class Momentum(SGD):
    def __init__(self, params, lr, wd=0., mom=0.9):
        super().__init__(params, lr=lr, wd=wd)
        self.mom=mom

    def opt_step(self, p):
        if not hasattr(p, 'grad_avg'): p.grad_avg = torch.zeros_like(p.grad)
        p.grad_avg = p.grad_avg*self.mom + p.grad*(1-self.mom)
        p -= self.lr * p.grad_avg

- to see the difference between this and the MomentumLearner implementation, run
`??MomentumLearner`
- with `MomentumLearner` we have:
  - more weight assigned to new data which makes the moving avg less smooth
  - we can compensate for that by increasing the batch size.
  - but increasing the batch size is [not a good practice](https://x.com/ylecun/status/989610208497360896?lang=en).
    - it makes the optimizer get stuck in sharp minima
    - it reduces the number of updates

In [ ]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(
    model, dls, F.cross_entropy,
    lr=1.5, # this is the 2nd time we can increase the `lr`. what was the 1st time?
    cbs=cbs, opt_func=Momentum
)
learn.fit(3)

In [ ]:
astats.color_dim()

## RMSProp

In [ ]:
class RMSProp(SGD):
    def __init__(self, params, lr, wd=0., sqr_mom=0.99, eps=1e-5):
        super().__init__(params, lr=lr, wd=wd)
        self.sqr_mom,self.eps = sqr_mom,eps

    def opt_step(self, p):
        if not hasattr(p, 'sqr_avg'):
          p.sqr_avg = p.grad**2
          # don't init to zero.
          # otherwise, the first step would be too big (divided by just `eps`)
        p.sqr_avg = p.sqr_avg*self.sqr_mom + p.grad**2*(1-self.sqr_mom)
        p -= self.lr * p.grad/(p.sqr_avg.sqrt() + self.eps)
        # we have to decrease the `lr` because of this division by a small number

In [ ]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=3e-3, cbs=cbs, opt_func=RMSProp)
learn.fit(3)

In [ ]:
astats.color_dim()

## Adam (RMSProp + Momentum)

In [ ]:
class Adam(SGD):
    def __init__(self, params, lr, wd=0., beta1=0.9, beta2=0.99, eps=1e-5):
        super().__init__(params, lr=lr, wd=wd)
        self.beta1,self.beta2,self.eps = beta1,beta2,eps

    def opt_step(self, p):
        if not hasattr(p, 'avg'): p.avg = torch.zeros_like(p.grad.data)
        if not hasattr(p, 'sqr_avg'): p.sqr_avg = torch.zeros_like(p.grad.data)
        p.avg = self.beta1*p.avg + (1-self.beta1)*p.grad
        # we unbias moving averages because we start from zero.
        # in each step `i` we're biased toward zero by beta**(i+1)
        # so we just use that value to unbias MAs
        unbias_avg = p.avg / (1 - (self.beta1**(self.i+1)))
        p.sqr_avg = self.beta2*p.sqr_avg + (1-self.beta2)*(p.grad**2)
        unbias_sqr_avg = p.sqr_avg / (1 - (self.beta2**(self.i+1)))
        p -= self.lr * unbias_avg / (unbias_sqr_avg + self.eps).sqrt()

In [ ]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=6e-3, cbs=cbs, opt_func=Adam)
learn.fit(3)

In [ ]:
astats.color_dim()

# Schedulers

- We've already seen how we can easily write a custom LR-adjusting callback or `Learner`
- Or can use the predefined PyTorch schedulers.
- We'll use the predefined ones for now since there's nothing new to learn in implementing them ourselves.

In [ ]:
sep = " | "
# print(sep.join(o for o in dir(lr_scheduler) if o[0].isupper() and o[1].islower()))
# print("==============")
def filter_fn(atr): return atr[0].isupper() and atr[1].islower() and ('LR' in atr)
print(sep.join(filter(filter_fn, dir(lr_scheduler))))

In [ ]:
learn = TrainLearner(get_model(), dls, F.cross_entropy, lr=6e-3, cbs=[DeviceCB(), SingleBatchCB()])
learn.fit(1)

# torch optimizer keep track of the parameter states in dicts
# it can have differen param groups
opt = learn.opt
param = next(iter(learn.model.parameters()))
st = opt.state[param]
print(st)
print("===========")
print(len(opt.param_groups))
print("===========")
pg = opt.param_groups[0]
print(list(pg))

# how a cosine scheduler works.
# note that after T_max=100 it increases the `lr`
sched = lr_scheduler.CosineAnnealingLR(opt, 100)
# print(sched.base_lrs)
# print(sched.get_last_lr())
# print("==================")
plt.figure(figsize=(4,4))
def sched_lrs(sched, steps):
    lrs = [sched.get_last_lr()]
    for i in range(steps):
        sched.optimizer.step()
        sched.step()
        lrs.append(sched.get_last_lr())
    plt.plot(lrs)

sched_lrs(sched, 110)

## Scheduler callbacks

In [ ]:
#|export
class BaseSchedCB(Callback):
  def __init__(self, sched): self.sched = sched
  def before_fit(self, learn): self.schedo = self.sched(learn.opt)
  def _step(self, learn):
    if learn.training: self.schedo.step()

class BatchSchedCB(BaseSchedCB):
  def after_batch(self, learn): self._step(learn)

class EpochSchedCB(BaseSchedCB):
  def after_epoch(self, learn): self._step(learn)

class HasLearnCB(Callback):
  def before_fit(self, learn): self.learn = learn
  def after_fit(self, learn): self.learn = None

class RecorderCB(Callback):
  def __init__(self, **d): self.d = d
  def before_fit(self, learn):
    self.recs = {k:[] for k in self.d}
    self.pg = learn.opt.param_groups[0]

  def after_batch(self, learn):
    if not learn.training: return
    for k,v in self.d.items():
      self.recs[k].append(v(self))

  def plot(self, fgs=(5,5)):
    for k,v in self.recs.items():
      plt.figure(figsize=fgs)
      plt.plot(v, label=k)
      plt.legend()
      plt.show()

In [ ]:
n_epochs = 3
sched = partial(
    lr_scheduler.CosineAnnealingLR,
    T_max=n_epochs * len(dls.train)
)
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
rec = RecorderCB(lr=lambda cb:cb.pg['lr'])
xtra = [BatchSchedCB(sched),rec]
learn = TrainLearner(model, dls, F.cross_entropy, lr=2e-2, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(n_epochs)

In [ ]:
rec.plot()

In [ ]:
sched = partial(lr_scheduler.CosineAnnealingLR, T_max=n_epochs)
set_seed(42)
xtra = [EpochSchedCB(sched),rec]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=2e-2, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(n_epochs)

In [ ]:
rec.plot()

## 1cycle training

- [Paper](https://arxiv.org/abs/1803.09820) by Leslie Smith.
  - It assumes that we always fail to init params correctly.
  - So, it starts the training with a warmup phase.
  - Then, it increases the `lr` to the max to speed up training
- If we can init params correctly, we actually don't need batch normalization or warmup
  - [fixup init](https://arxiv.org/abs/1901.09321) paper
  - [T-fixup](https://www.cs.toronto.edu/~mvolkovs/ICML2020_tfixup.pdf) paper

In [ ]:
rec = RecorderCB(
    lr=lambda cb:cb.pg['lr'],
    mom=lambda cb: cb.pg['betas'][0]
)

set_seed(42)
lr,n_epochs = 6e-2,5
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
sched = partial(
    lr_scheduler.OneCycleLR,
    max_lr=lr,
    total_steps=n_epochs * len(dls.train)
)
xtra = [BatchSchedCB(sched), rec]
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(n_epochs)

In [ ]:
rec.plot()